# Structured output from LLMs — companion notebook

**Course:** AI RAG Agents — Meeting 10-11

This notebook walks through four ways to get **structured, typed data** out of an LLM, in increasing order of reliability and ergonomics.

1. **Prompt-only JSON** — works, fails fragile-ly, good baseline
2. **Native OpenAI tool calling** — the raw mechanism, no framework
3. **`instructor` + Pydantic** — the same task, much less code
4. **Validators + auto-retry** — domain rules that fix themselves

At the end there's a small **"your turn"** exercise.

---

### Setup

We need `openai`, `pydantic`, `instructor`, and `python-dotenv`. Your `.env` (at `/home/amir/source/.env`) should contain `OPENAI_API_KEY`.

In [ ]:
# Install once if needed
# %pip install -q openai pydantic instructor python-dotenv

In [1]:
import json
import os
from dotenv import load_dotenv

load_dotenv("/home/amir/.env")

MODEL = "gpt-4o-mini"  # cheap + capable enough for everything below
print("OK")

OK


## A shared toy task

Every demo extracts the **same target** from the **same input** so we can compare apples to apples.

**Input:** a short product review.

**Target schema:**
- `sentiment` — one of `positive`, `negative`, `neutral`, `mixed`
- `confidence` — float in `[0, 1]`
- `aspects` — list of (aspect, polarity) pairs found in the review

In [2]:
REVIEW = (
    "The battery life is incredible — easily two full days. "
    "But the camera is a letdown in low light, and the price is steep. "
    "Build quality is solid though."
)

---
## Demo 1 — Prompt-only JSON

The lowest-tech approach: just ask politely for JSON and hope. Then parse it ourselves.

This is the right baseline. It works on **any** model, including ones with no tool-calling support.

Watch what can go wrong:
- The model wraps the JSON in ```` ```json ```` fences.
- The model adds a chatty preamble ("Sure! Here's the JSON:").
- The model invents extra keys, or omits required ones.
- A typo makes the JSON unparseable.

In [3]:
from openai import OpenAI

client = OpenAI()

PROMPT_ONLY_INSTRUCTIONS = """\
You are a sentiment analyzer.

Return ONLY a JSON object — no prose, no code fences — with these keys:
- sentiment: one of "positive", "negative", "neutral", "mixed"
- confidence: a number between 0 and 1
- aspects: a list of objects, each with "name" (string) and "polarity"
  (one of "positive", "negative")
"""

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": PROMPT_ONLY_INSTRUCTIONS},
        {"role": "user", "content": REVIEW},
    ],
)
raw = resp.choices[0].message.content
print("--- raw response ---")
print(raw)

--- raw response ---
{
  "sentiment": "mixed",
  "confidence": 0.85,
  "aspects": [
    {
      "name": "battery life",
      "polarity": "positive"
    },
    {
      "name": "camera",
      "polarity": "negative"
    },
    {
      "name": "price",
      "polarity": "negative"
    },
    {
      "name": "build quality",
      "polarity": "positive"
    }
  ]
}


In [4]:
# Naive parse — works most of the time, fails the rest
try:
    parsed = json.loads(raw)
    print("parsed OK:", parsed)
except json.JSONDecodeError as e:
    print("JSON parse failed:", e)
    print("You'd now write fence-stripping code, retry logic, etc.")

parsed OK: {'sentiment': 'mixed', 'confidence': 0.85, 'aspects': [{'name': 'battery life', 'polarity': 'positive'}, {'name': 'camera', 'polarity': 'negative'}, {'name': 'price', 'polarity': 'negative'}, {'name': 'build quality', 'polarity': 'positive'}]}


**Try this:** rerun the cell above a few times. On `gpt-4o-mini` with this prompt it's usually clean, but switch to a smaller / older model and you'll see fences and prose appear.

**JSON mode** (one step up) guarantees valid JSON but still doesn't enforce the *shape*:

```python
client.chat.completions.create(
    model=MODEL,
    response_format={"type": "json_object"},   # <- valid JSON guaranteed
    messages=[...],
)
```

You still need to validate keys / types yourself.

---
## Demo 2 — Native OpenAI tool calling (raw SDK)

This is the mechanism every "structured output" framework wraps. Worth seeing once.

We define a function ("tool") with a JSON Schema. We tell the model it **must** call this tool. The provider enforces the schema and returns the arguments as JSON we can parse.

In [5]:
from pydantic import BaseModel, Field
from typing import Literal

class Aspect(BaseModel):
    name: str = Field(description="The product aspect mentioned, e.g., 'battery'")
    polarity: Literal["positive", "negative"]

class Sentiment(BaseModel):
    sentiment: Literal["positive", "negative", "neutral", "mixed"] = Field(
        description="Overall sentiment of the review"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="How confident you are, 0 to 1",
    )
    aspects: list[Aspect] = Field(
        description="Specific product aspects mentioned, with their polarity"
    )

# Pydantic generates the JSON Schema for us
schema = Sentiment.model_json_schema()
print(json.dumps(schema, indent=2)[:600], "...")

{
  "$defs": {
    "Aspect": {
      "properties": {
        "name": {
          "description": "The product aspect mentioned, e.g., 'battery'",
          "title": "Name",
          "type": "string"
        },
        "polarity": {
          "enum": [
            "positive",
            "negative"
          ],
          "title": "Polarity",
          "type": "string"
        }
      },
      "required": [
        "name",
        "polarity"
      ],
      "title": "Aspect",
      "type": "object"
    }
  },
  "properties": {
    "sentiment": {
      "description": "Overall sentiment of the revi ...


In [6]:
tools = [{
    "type": "function",
    "function": {
        "name": "record_sentiment",
        "description": "Record the structured sentiment analysis of a product review.",
        "parameters": schema,
    },
}]

resp = client.chat.completions.create(
    model=MODEL,
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "record_sentiment"}},
    messages=[{"role": "user", "content": REVIEW}],
)

tool_call = resp.choices[0].message.tool_calls[0]
args_json = tool_call.function.arguments
print("--- raw arguments string ---")
print(args_json)

--- raw arguments string ---
{"sentiment":"mixed","confidence":0.8,"aspects":[{"name":"battery life","polarity":"positive"},{"name":"camera","polarity":"negative"},{"name":"price","polarity":"negative"},{"name":"build quality","polarity":"positive"}]}


In [7]:
# Parse + Pydantic-validate in one step
result = Sentiment.model_validate_json(args_json)
print(type(result).__name__, "->", result)
print()
print("sentiment :", result.sentiment)
print("confidence:", result.confidence)
for a in result.aspects:
    print(f"  - {a.name:<10s} {a.polarity}")

Sentiment -> sentiment='mixed' confidence=0.8 aspects=[Aspect(name='battery life', polarity='positive'), Aspect(name='camera', polarity='negative'), Aspect(name='price', polarity='negative'), Aspect(name='build quality', polarity='positive')]

sentiment : mixed
confidence: 0.8
  - battery life positive
  - camera     negative
  - price      negative
  - build quality positive


Notice what we wrote:
- A Pydantic model.
- A tool definition wrapping its JSON Schema.
- `tool_choice` to **force** the call (so the model can't decide to chat instead).
- A parse step.

Every framework below replaces those four steps with one argument: `response_model=Sentiment`.

---
## Demo 3 — `instructor` + Pydantic

Same task, ergonomic API. `instructor` handles tool definition, forcing the call, parsing, and (in Demo 4) retries.


In [8]:
import instructor

iclient = instructor.from_openai(OpenAI())

result: Sentiment = iclient.chat.completions.create(
    model=MODEL,
    response_model=Sentiment,
    messages=[{"role": "user", "content": REVIEW}],
)

print(result)
print()
print("as dict:", result.model_dump())

sentiment='mixed' confidence=0.85 aspects=[Aspect(name='battery life', polarity='positive'), Aspect(name='camera', polarity='negative'), Aspect(name='price', polarity='negative'), Aspect(name='build quality', polarity='positive')]

as dict: {'sentiment': 'mixed', 'confidence': 0.85, 'aspects': [{'name': 'battery life', 'polarity': 'positive'}, {'name': 'camera', 'polarity': 'negative'}, {'name': 'price', 'polarity': 'negative'}, {'name': 'build quality', 'polarity': 'positive'}]}


That's the entire program. Same Pydantic model, same model, same review — five lines instead of fifteen.

### Field descriptions are prompts — a quick A/B

Watch how much the *description* on a single field matters.

In [9]:
class CitationVague(BaseModel):
    author: str = Field(description="author")
    year: int = Field(description="year")

class CitationSharp(BaseModel):
    author: str = Field(
        description="Last name of the FIRST author only, no initials, no 'et al.'"
    )
    year: int = Field(
        description="4-digit publication year. If only a decade is given (e.g., 'the 90s'), use the first year of that decade."
    )

AMBIGUOUS = "As Smith and colleagues showed back in the late 90s, transformers were inevitable."

vague  = iclient.chat.completions.create(
    model=MODEL, response_model=CitationVague,
    messages=[{"role": "user", "content": AMBIGUOUS}],
)
sharp  = iclient.chat.completions.create(
    model=MODEL, response_model=CitationSharp,
    messages=[{"role": "user", "content": AMBIGUOUS}],
)

print("vague :", vague)
print("sharp :", sharp)

vague : author='Smith et al.' year=1999
sharp : author='Smith' year=1997


The `vague` version often returns inconsistent author formats and unstable years on the decade case. The `sharp` version is far more deterministic. **The schema is a prompt.**

---
## Demo 4 — Validators + auto-retry

Pydantic validators run after parsing. If they raise, `instructor` will **re-ask the model with the error message** as additional context, up to `max_retries` times.

Use this for *domain* rules JSON Schema can't express on its own: implausible years, mismatched fields, business constraints.

In [10]:
from pydantic import field_validator, model_validator

class StrictCitation(BaseModel):
    author: str = Field(description="Last name of the first author only")
    year: int = Field(description="4-digit publication year")

    @field_validator("author")
    @classmethod
    def author_no_initials_or_etal(cls, v: str) -> str:
        v = v.strip()
        if "." in v or "et al" in v.lower() or " " in v:
            raise ValueError(
                f"Author '{v}' must be a single last name only, with no initials and no 'et al.'"
            )
        return v

    @field_validator("year")
    @classmethod
    def year_must_be_plausible(cls, v: int) -> int:
        if not (1900 <= v <= 2030):
            raise ValueError(f"Year {v} is implausible — must be between 1900 and 2030")
        return v

In [11]:
# Deliberately ambiguous input that often trips the model on the first try
TRICKY = "In the seminal 'Attention is All You Need' paper, Vaswani et al. (2017) introduced..."

result = iclient.chat.completions.create(
    model=MODEL,
    response_model=StrictCitation,
    max_retries=2,    # <- the magic. Validation errors are sent back to the model.
    messages=[{"role": "user", "content": TRICKY}],
)
print(result)

author='Vaswani' year=2017


Even when the first attempt comes back as `"Vaswani et al."` (and our validator rejects it), `instructor` re-prompts the model with the validation error — and the second attempt usually returns just `"Vaswani"`.

**Lesson:** validators turn schema enforcement into *self-healing* extraction. But cap `max_retries` low (1–2) — if you need more, fix the prompt or schema upstream.

### Cross-field validation with `model_validator`

In [12]:
class DateRange(BaseModel):
    start_year: int
    end_year: int

    @model_validator(mode="after")
    def end_after_start(self):
        if self.end_year < self.start_year:
            raise ValueError(
                f"end_year ({self.end_year}) must be >= start_year ({self.start_year})"
            )
        return self

result = iclient.chat.completions.create(
    model=MODEL,
    response_model=DateRange,
    max_retries=2,
    messages=[{"role": "user", "content": "The study ran from 2019 to 2022."}],
)
print(result)

start_year=2019 end_year=2022


---
## Bonus — A pattern you'll reuse: routing

Classification with an enum is the simplest, most reliable form of structured output. Agents use this to pick which tool to call.

In [13]:
class Route(BaseModel):
    destination: Literal["sql_db", "vector_store", "web_search", "none"] = Field(
        description=(
            "Where to route the query. "
            "sql_db: structured tabular questions (counts, aggregates). "
            "vector_store: semantic questions about our internal docs. "
            "web_search: current events or things outside our docs. "
            "none: small talk or refusals."
        )
    )
    reason: str = Field(description="One sentence on why this route.")

for q in [
    "How many active users did we have last month?",
    "What does our onboarding doc say about SSO?",
    "Who won the F1 race yesterday?",
    "hi there!",
]:
    r = iclient.chat.completions.create(
        model=MODEL, response_model=Route,
        messages=[{"role": "user", "content": q}],
    )
    print(f"{q!r:<55s} -> {r.destination:<14s} ({r.reason})")

'How many active users did we have last month?'         -> sql_db         (To retrieve the count of active users for the previous month.)
'What does our onboarding doc say about SSO?'           -> vector_store   (Looking for specific information about SSO in the onboarding document.)
'Who won the F1 race yesterday?'                        -> web_search     (Looking for current events related to the F1 race results.)
'hi there!'                                             -> none           (The user is initiating a casual conversation.)


---
## Your turn — exercise

Pick an arxiv abstract (you can grab one from the `jamescalam/ai-arxiv-chunked` dataset we use elsewhere in the course, or paste any abstract you like).

Build a `PaperMetadata` Pydantic model with **at least** these fields:

- `title: str`
- `authors: list[str]` — last names only
- `year: int | None` — `None` if not stated
- `task: Literal[...]` — pick a closed set of NLP/CV tasks
- `keywords: list[str]` — capped at 5

Then:

1. Add `Field(description=...)` to **every** field. Make them sharp.
2. Add a `field_validator` that rejects authors containing initials or commas.
3. Use `instructor` with `max_retries=2` to extract from your abstract.
4. Run it on **two** different abstracts. Compare. Where does it struggle?
5. Try removing the descriptions. How does quality change?

**Stretch goal:** add a `summary: str` field with a description that asks for a one-sentence summary in **your own words** (not copied from the abstract). Inspect a few outputs — is the model actually rewriting, or just paraphrasing the first sentence?

In [14]:
# Your code here
ABSTRACT = """\
Paste an abstract here.
"""

# class PaperMetadata(BaseModel):
#     ...

# meta = iclient.chat.completions.create(
#     model=MODEL,
#     response_model=PaperMetadata,
#     max_retries=2,
#     messages=[{"role": "user", "content": ABSTRACT}],
# )
# print(meta)

---
## Recap

- **Prompt-only** is a fine baseline; ergonomics fall apart fast.
- **Tool calling** is the modern default for hosted models — schema-enforced, reliable.
- **Pydantic + `instructor`** turns the whole thing into one function call.
- **Field descriptions are prompts** — they're the highest-leverage thing you write.
- **Validators + bounded retries** give you self-healing extraction without hiding bugs.
- **Patterns**: extraction, routing, agent plans, eval rubrics — all the same shape.

Next meeting we'll plug this into a real agent loop.